# 1D CNN for Beginners 🧠

A **1D Convolutional Neural Network (1D CNN)** slides small filters across a *sequence* (like a time series, sensor signal, or ECG reading) to automatically detect useful local patterns.

**When to use a 1D CNN?**
- Time-series classification (activity recognition, ECG/EEG signals)
- Sensor data (accelerometer, audio waveforms)
- Any 1D signal where *local shape* matters

**How it differs from a 2D CNN:** a 2D CNN slides a filter over an image (height × width). A 1D CNN slides a filter over a single axis — time/position.

In this notebook we'll:
1. Generate a simple synthetic signal dataset
2. Understand the shape a 1D CNN expects
3. Build, train, and evaluate a small 1D CNN
4. Look at the predictions

> No prior deep-learning experience needed. Each step is explained.


## 1. Setup

We use TensorFlow/Keras. If it isn't installed, uncomment the pip line.

In [ ]:
# !pip install tensorflow scikit-learn matplotlib

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

np.random.seed(42)
tf.random.set_seed(42)
print("TensorFlow version:", tf.__version__)

## 2. Create a synthetic dataset

We'll build a **binary classification** problem. Each sample is a 1D signal of length 100.

- **Class 0:** a noisy *sine* wave
- **Class 1:** a noisy *square* wave

The network's job is to learn which shape it's looking at. This is a great toy problem because the two classes have clearly different *local patterns* — exactly what a 1D CNN is good at picking up.

In [ ]:
def make_dataset(n_per_class=500, length=100):
    X, y = [], []
    t = np.linspace(0, 4 * np.pi, length)

    for _ in range(n_per_class):
        # Class 0: sine wave + noise
        freq = np.random.uniform(0.8, 1.2)
        sine = np.sin(freq * t) + np.random.normal(0, 0.2, length)
        X.append(sine); y.append(0)

        # Class 1: square wave + noise
        freq = np.random.uniform(0.8, 1.2)
        square = np.sign(np.sin(freq * t)) + np.random.normal(0, 0.2, length)
        X.append(square); y.append(1)

    X = np.array(X, dtype="float32")
    y = np.array(y, dtype="float32")
    return X, y

X, y = make_dataset()
print("X shape:", X.shape)   # (n_samples, length)
print("y shape:", y.shape)

Let's plot one example from each class so we can *see* what the model needs to distinguish.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].plot(X[0]); axes[0].set_title("Class 0: sine")
axes[1].plot(X[1]); axes[1].set_title("Class 1: square")
for ax in axes: ax.set_xlabel("time step"); ax.set_ylabel("value")
plt.tight_layout(); plt.show()

## 3. Reshape for the 1D CNN

A Keras `Conv1D` layer expects input shaped as:

```
(batch_size, sequence_length, num_channels)
```

Our signal has **1 channel** (a single value per time step), so we add a channel dimension at the end.

Think of "channels" like the R/G/B channels of an image — here we only have one measurement per time step, so there's just one channel.

In [ ]:
X = X[..., np.newaxis]   # (n_samples, 100, 1)
print("New X shape:", X.shape)

### Train / test split
We hold out 20% of the data to check the model generalizes to signals it hasn't seen.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 4. Build the 1D CNN

Our small architecture:

| Layer | What it does |
|-------|--------------|
| `Conv1D(16, 3)` | Slides 16 filters of width 3 to detect small local patterns |
| `MaxPooling1D(2)` | Downsamples by taking the max over every 2 steps (keeps strong signals, shrinks size) |
| `Conv1D(32, 3)` | A deeper layer that combines earlier patterns into richer ones |
| `GlobalMaxPooling1D` | Collapses the whole sequence into one vector per filter |
| `Dense(1, sigmoid)` | Outputs a probability between 0 and 1 |

`relu` activations add non-linearity so the network can learn complex shapes.

In [ ]:
model = models.Sequential([
    layers.Input(shape=(100, 1)),
    layers.Conv1D(filters=16, kernel_size=3, activation="relu"),
    layers.MaxPooling1D(pool_size=2),
    layers.Conv1D(filters=32, kernel_size=3, activation="relu"),
    layers.GlobalMaxPooling1D(),
    layers.Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## 5. Train the model

- **Loss** = `binary_crossentropy` — the standard choice for 2-class problems.
- **Optimizer** = `adam` — a reliable default that adjusts learning rates automatically.
- **Epochs** = 15 — one epoch is one full pass over the training data.

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=15,
    batch_size=32,
    verbose=1,
)

### Plot the learning curves
Watching training vs. validation accuracy tells us if the model is learning well and not badly overfitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="val")
axes[0].set_title("Accuracy"); axes[0].set_xlabel("epoch"); axes[0].legend()

axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="val")
axes[1].set_title("Loss"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

## 6. Evaluate on the test set
This is the honest measure of performance — data the model never trained on.

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.3f}")
print(f"Test loss:     {test_loss:.3f}")

### Inspect a few predictions

In [ ]:
probs = model.predict(X_test[:6], verbose=0).ravel()

fig, axes = plt.subplots(1, 6, figsize=(16, 2.5))
for i, ax in enumerate(axes):
    ax.plot(X_test[i].ravel())
    pred = int(probs[i] > 0.5)
    ax.set_title(f"true={int(y_test[i])}\npred={pred} ({probs[i]:.2f})",
                 color="green" if pred == int(y_test[i]) else "red")
    ax.set_xticks([])
plt.tight_layout(); plt.show()

## 7. What you learned & next steps

**Recap**
- A 1D CNN slides filters across a sequence to detect local patterns.
- Input must be shaped `(batch, sequence_length, channels)`.
- The same build → compile → fit → evaluate workflow applies to any Keras model.

**Try next**
1. Add a third class (e.g. a sawtooth wave) → switch the final layer to `Dense(3, activation="softmax")` and the loss to `sparse_categorical_crossentropy`.
2. Increase noise and see how accuracy responds.
3. Swap in a **real** dataset — the [UCI Human Activity Recognition](https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones) accelerometer data is a classic 1D-CNN benchmark.
4. Add `layers.Dropout(0.3)` before the final layer to reduce overfitting.

Happy experimenting! 🚀